# Inicializando


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import col, trim, to_date, length

# Lendo dados da Tabela Bronze

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")
df.display()

# Remoção de espaços desnecessários de colunas string

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

# Tratamento de zero e nulos em colunas de data

In [0]:
%python

df = (
    df.withColumn("sls_order_dt", F.when((F.col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8) , None).otherwise(F.col("sls_order_dt")))
    .withColumn("sls_ship_dt", F.when((F.col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8), None).otherwise(F.col("sls_ship_dt")))
    .withColumn("sls_due_dt", F.when((F.col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8), None).otherwise(F.col("sls_due_dt")))
)


df.display()

# Transformando colunas para tipo Date

In [0]:
df = df.withColumn("sls_order_dt", to_date(col("sls_order_dt"), "yyyyMMdd"))
df = df.withColumn("sls_ship_dt", to_date(col("sls_ship_dt"), "yyyyMMdd"))
df = df.withColumn("sls_due_dt", to_date(col("sls_due_dt"), "yyyyMMdd"))

df.display()

# Tratamento de valores nulos na coluna sls_price

In [0]:
df = (
    df.withColumn(
        'sls_price', 
        F.when(
            (F.col('sls_price').isNull()) | (F.col('sls_price') <= 0), 
            F.when(
                F.col('sls_quantity') != 0, 
                F.col('sls_sales') / F.col('sls_quantity') 
            ).otherwise(None)
        ).otherwise(F.col('sls_price'))
    )
)


df.display()

# Renomear Colunas

In [0]:
RENAME_MAP = {
    'sls_ord_num': 'order_number',
    'sls_prd_key': 'product_key',
    'sls_cust_id': 'customer_id',
    'sls_order_dt': 'order_date',
    'sls_ship_dt': 'ship_date',
    'sls_due_dt': 'due_date',
    'sls_sales': 'sales',
    'sls_quantity': 'quantity',
    'sls_price': 'price'
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.limit(10).display()

# Criando Tabela Silver

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable('workspace.silver.crm_sales')

# Verificação de Dados na Tabela final

In [0]:
%sql

SELECT * FROM workspace.silver.crm_sales LIMIT 10